## 1. Import Libraries

This section imports the required Python libraries for data cleaning and analysis.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 2. Load Raw Dataset

The dataset is loaded from the `data` folder.

Child ID is loaded as a string to avoid scientific notation problems.


In [2]:
# Possible file locations
possible_paths = [
    Path("../data/GmpData 2.xlsx"),
    Path("../data/GmpData 2(1).xlsx"),
    Path("data/GmpData 2.xlsx"),
    Path("GmpData 2.xlsx")
]

DATA_PATH = None

for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError("GmpData 2.xlsx file not found. Please check the data folder.")

print("Using file:", DATA_PATH)

df = pd.read_excel(DATA_PATH, dtype={"ID": str})

raw_records = len(df)
raw_children = df["ID"].nunique(dropna=True)

print("Raw Records:", raw_records)
print("Raw Unique Children:", raw_children)

df.head()


Using file: ..\data\GmpData 2.xlsx
Raw Records: 10011
Raw Unique Children: 3550


,ID,Name,DOB,GmpDate,MUAC,Age
0,912031130025,abusaid,2016-06-07,2017-12-11,14.0,102.0
1,912031130016,sompa,2017-04-16,2017-12-11,13.0,92.0
2,912031140018,Masuma,2016-10-04,2017-11-14,0.0,98.0
3,912031140018,Masuma,2016-10-04,2017-12-11,1.0,98.0
4,912031140018,Masuma,2016-10-04,2017-12-11,11.0,98.0


## 3. Basic Dataset Information

This section checks the dataset shape, column names, data types, and unique child count.


In [3]:
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nUnique Children:", df["ID"].nunique(dropna=True))


Dataset Shape: (10011, 6)
Columns: ['ID', 'Name', 'DOB', 'GmpDate', 'MUAC', 'Age']

Data Types:
ID                    str
Name                  str
DOB        datetime64[us]
GmpDate    datetime64[us]
MUAC              float64
Age               float64
dtype: object

Unique Children: 3550


## 4. Missing Value Check

This section checks missing values in each column.


In [4]:
missing_values = df.isnull().sum()
missing_values


ID          11
Name         5
DOB        133
GmpDate     11
MUAC        39
Age        133
dtype: int64

## 5. Clean ID and Name Columns

Child ID and Name are cleaned by removing extra spaces.

Empty or invalid names are converted into missing values.


In [5]:
df["ID"] = df["ID"].astype(str).str.strip()

df["Name"] = df["Name"].astype(str).str.strip()
df["Name"] = df["Name"].replace(
    ["", " ", "nan", "NaN", "NA", "N/A", "NULL", "null"],
    np.nan
)

df[["ID", "Name"]].head()


,ID,Name
0,912031130025,abusaid
1,912031130016,sompa
2,912031140018,Masuma
3,912031140018,Masuma
4,912031140018,Masuma


## 6. Convert Date Columns

DOB and GmpDate are converted into datetime format.

Invalid date values are converted into missing date values.


In [6]:
df["DOB"] = pd.to_datetime(df["DOB"], errors="coerce")
df["GmpDate"] = pd.to_datetime(df["GmpDate"], errors="coerce")

print(df[["DOB", "GmpDate"]].dtypes)
df[["DOB", "GmpDate"]].head()


DOB        datetime64[us]
GmpDate    datetime64[us]
dtype: object


,DOB,GmpDate
0,2016-06-07,2017-12-11
1,2017-04-16,2017-12-11
2,2016-10-04,2017-11-14
3,2016-10-04,2017-12-11
4,2016-10-04,2017-12-11


## 7. Convert Numeric Columns

MUAC and Age are converted into numeric format.

Invalid numeric values are converted into missing values.


In [7]:
df["MUAC"] = pd.to_numeric(df["MUAC"], errors="coerce")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

print(df[["MUAC", "Age"]].dtypes)
df[["MUAC", "Age"]].describe()


MUAC    float64
Age     float64
dtype: object


,MUAC,Age
count,9972.000000,9878.000000
mean,12.554917,108.520045
std,24.849072,141.083120
min,0.000000,-89.000000
25%,12.000000,86.000000
50%,13.200000,95.000000
75%,14.200000,100.000000
max,1302.000000,1499.000000


## 8. Remove Missing Critical Records

Records without Child ID, GMP date, DOB, or MUAC cannot be used for this analysis.

These records are removed.


In [8]:
before_missing_clean = len(df)

df = df.dropna(
    subset=["ID", "DOB", "GmpDate", "MUAC"]
).copy()

after_missing_clean = len(df)

print("Records before missing cleaning:", before_missing_clean)
print("Records after missing cleaning:", after_missing_clean)
print("Removed records:", before_missing_clean - after_missing_clean)


Records before missing cleaning: 10011
Records after missing cleaning: 9850
Removed records: 161


## 9. Age Calculation from DOB and GMP Date

The original Age column contains unrealistic values.

Therefore, a new age feature is calculated from DOB and GmpDate.

Age is calculated in months.


In [9]:
df["Age_Months"] = (
    (df["GmpDate"] - df["DOB"]).dt.days / 30.44
).round(2)

df["Age_Diff"] = (
    df["Age"] - df["Age_Months"]
).abs().round(2)

print("Original Age Min:", df["Age"].min())
print("Original Age Max:", df["Age"].max())

print("\nCalculated Age Summary:")
df["Age_Months"].describe()


Original Age Min: -89.0
Original Age Max: 1499.0

Calculated Age Summary:


count    9850.000000
mean       25.678947
std       141.920788
min      -172.110000
25%         3.350000
50%        11.990000
75%        17.810000
max      1434.000000
Name: Age_Months, dtype: float64

## 10. Remove Invalid Age Records

Records with negative age or extremely high age are invalid.

For this research dataset, valid age is kept between 0 and 120 months.


In [10]:
invalid_age = df[
    (df["Age_Months"] < 0) |
    (df["Age_Months"] > 120)
]

print("Invalid Age Records:", len(invalid_age))

invalid_age[
    ["ID", "Name", "DOB", "GmpDate", "MUAC", "Age", "Age_Months", "Age_Diff"]
].head(20)


Invalid Age Records: 449


,ID,Name,DOB,GmpDate,MUAC,Age,Age_Months,Age_Diff
25,912031120022,jafuan,2016-09-05,2016-05-09,14.0,99.0,-3.91,102.91
116,912031010012,Babi,2017-12-19,2017-12-17,12.0,84.0,-0.07,84.07
125,910813010044,g,2017-12-18,2017-12-17,11.0,84.0,-0.03,84.03
131,581415010016,Showy Ullah,1990-12-13,2017-12-17,13.5,408.0,324.11,83.89
145,587480010008,Saddam,2017-06-06,2017-02-11,13.0,90.0,-3.78,93.78
146,587480010008,Saddam,2017-06-06,2017-04-25,13.5,90.0,-1.38,91.38
149,587480010005,Moriom,2017-12-18,2017-12-17,14.0,84.0,-0.03,84.03
170,910813010025,Fahim,2017-12-20,2017-12-17,2.0,84.0,-0.10,84.10
193,912031010020,q,2017-12-17,2017-12-16,12.0,84.0,-0.03,84.03
220,919447010020,razia,2017-10-16,2017-05-18,14.0,86.0,-4.96,90.96


In [11]:
before_age_clean = len(df)

df = df[
    (df["Age_Months"] >= 0) &
    (df["Age_Months"] <= 120)
].copy()

after_age_clean = len(df)

print("Records Before Age Validation:", before_age_clean)
print("Records After Age Validation:", after_age_clean)
print("Removed Invalid Age Records:", before_age_clean - after_age_clean)

df["Age_Months"].describe()


Records Before Age Validation: 9850
Records After Age Validation: 9401
Removed Invalid Age Records: 449


count    9401.000000
mean       11.709724
std         8.640409
min         0.000000
25%         4.140000
50%        11.990000
75%        17.810000
max       112.090000
Name: Age_Months, dtype: float64

## 11. MUAC Validation

MUAC values outside a realistic range are considered invalid.

For this dataset, MUAC values between 5 cm and 30 cm are kept.


In [12]:
invalid_muac = df[
    (df["MUAC"] < 5) |
    (df["MUAC"] > 30)
]

print("Invalid MUAC Records:", len(invalid_muac))

invalid_muac[
    ["ID", "Name", "DOB", "GmpDate", "MUAC", "Age_Months"]
].head(20)


Invalid MUAC Records: 1512


,ID,Name,DOB,GmpDate,MUAC,Age_Months
2,912031140018,Masuma,2016-10-04,2017-11-14,0.0,13.34
3,912031140018,Masuma,2016-10-04,2017-12-11,1.0,14.22
5,912031140018,Masuma,2016-10-04,2018-02-12,0.0,16.29
9,912031140002,Maria,2016-10-04,2017-12-11,1.0,14.22
10,912031140002,Maria,2016-10-04,2018-02-12,0.0,16.29
55,912031140011,lion,2016-01-31,2017-11-14,148.0,21.45
85,912021010018,sabuj,2017-08-10,2017-12-11,1.0,4.04
86,912021010002,maria,2017-10-04,2017-12-11,1.0,2.23
99,916238010005,g gcfc,2017-12-15,2017-12-15,0.0,0.00
100,587480010001,y,2017-12-16,2017-12-16,555.0,0.00


In [13]:
before_muac_clean = len(df)

df = df[
    (df["MUAC"] >= 5) &
    (df["MUAC"] <= 30)
].copy()

after_muac_clean = len(df)

print("Records Before MUAC Cleaning:", before_muac_clean)
print("Records After MUAC Cleaning:", after_muac_clean)
print("Removed Invalid MUAC Records:", before_muac_clean - after_muac_clean)

df["MUAC"].describe()


Records Before MUAC Cleaning: 9401
Records After MUAC Cleaning: 7889
Removed Invalid MUAC Records: 1512


count    7889.000000
mean       13.577394
std         2.181080
min         5.000000
25%        12.900000
50%        13.700000
75%        14.500000
max        30.000000
Name: MUAC, dtype: float64

## 12. Data Quality Audit: Same ID, Different Name

This section checks whether the same child ID has multiple names.

For research purpose, Child ID is considered the main identifier.
Name conflicts are treated as data entry inconsistencies.


In [14]:
name_conflict_before = (
    df.groupby("ID")["Name"]
    .nunique(dropna=True)
    .gt(1)
    .sum()
)

print("IDs with multiple names before standardization:", name_conflict_before)

name_conflict_examples = (
    df.groupby("ID")["Name"]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)

name_conflict_examples.head(20)


IDs with multiple names before standardization: 515


ID
913836010010    16
913836010002    11
913836010001     8
913836010009     8
588042010025     8
913836010005     7
913836010003     7
912031010005     7
915938010001     6
915919010009     6
910813010001     6
919447010001     6
911711010001     6
916238120006     6
915947150001     6
911711010005     6
919447010005     6
588042010024     6
913836010008     5
912031010002     5
Name: Name, dtype: int64

## 13. Data Quality Audit: Same ID, Different DOB

This section checks whether the same child ID has multiple DOB values.

Instead of removing these child records, DOB is standardized using the most frequent DOB for each child ID.


In [15]:
dob_conflict_before = (
    df.groupby("ID")["DOB"]
    .nunique(dropna=True)
    .gt(1)
    .sum()
)

print("IDs with multiple DOB before standardization:", dob_conflict_before)

dob_conflict_examples = (
    df.groupby("ID")["DOB"]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)

dob_conflict_examples.head(20)


IDs with multiple DOB before standardization: 387


ID
913836010010    9
912031010005    7
913836010002    7
913836010001    7
913836010003    6
913836010009    6
910813010001    6
919447010001    6
588042010024    6
588042010025    6
913836010005    6
911711010005    5
913836010012    5
915938010001    5
910867050001    5
912031010025    5
910927010012    5
916238010010    4
911711010003    4
911711010023    4
Name: DOB, dtype: int64

## 14. Standardize Name and DOB by Child ID

For the same child ID, the most frequent Name and DOB are selected.

Original Name and DOB are preserved in separate columns for audit purpose.


In [16]:
def most_frequent_value(series):
    series = series.dropna()

    if len(series) == 0:
        return np.nan

    return series.mode().iloc[0]


# Preserve original values for audit
df["Name_Original"] = df["Name"]
df["DOB_Original"] = df["DOB"]

# Create most frequent value mapping for each child ID
name_map = df.groupby("ID")["Name"].apply(most_frequent_value).to_dict()
dob_map = df.groupby("ID")["DOB"].apply(most_frequent_value).to_dict()

# Apply standardized values
df["Name"] = df["ID"].map(name_map)
df["DOB"] = df["ID"].map(dob_map)

# Recalculate age after DOB standardization
df["Age_Months"] = (
    (df["GmpDate"] - df["DOB"]).dt.days / 30.44
).round(2)

df["Age_Diff"] = (
    df["Age"] - df["Age_Months"]
).abs().round(2)

# Check remaining conflicts
name_conflict_after = (
    df.groupby("ID")["Name"]
    .nunique(dropna=True)
    .gt(1)
    .sum()
)

dob_conflict_after = (
    df.groupby("ID")["DOB"]
    .nunique(dropna=True)
    .gt(1)
    .sum()
)

print("Name conflicts after standardization:", name_conflict_after)
print("DOB conflicts after standardization:", dob_conflict_after)


Name conflicts after standardization: 0
DOB conflicts after standardization: 0


## 15. Revalidate Age after DOB Standardization

After DOB standardization, age is recalculated again.

Invalid age records are removed again if created by DOB correction.


In [17]:
invalid_age_after_standardization = df[
    (df["Age_Months"] < 0) |
    (df["Age_Months"] > 120)
]

print("Invalid Age Records After Standardization:", len(invalid_age_after_standardization))

invalid_age_after_standardization[
    ["ID", "Name", "DOB", "DOB_Original", "GmpDate", "MUAC", "Age_Months"]
].head(20)


Invalid Age Records After Standardization: 65


,ID,Name,DOB,DOB_Original,GmpDate,MUAC,Age_Months
87,912021010003,p,2017-12-31,2016-11-19,2017-12-11,14.0,-0.66
101,912031010002,Arnob,2017-12-18,2017-11-01,2017-12-16,15.0,-0.07
111,912031010010,MALIYAT,2017-12-18,2017-12-17,2017-12-17,13.0,-0.03
117,913836010001,NIROB,2018-01-08,2017-04-30,2017-12-14,11.5,-0.82
132,587480010002,Asma,2017-12-18,2017-12-17,2017-12-17,10.0,-0.03
136,913836010001,NIROB,2018-01-08,2016-12-17,2017-12-17,10.0,-0.72
159,588042010024,Naeem,2017-12-27,2016-11-16,2017-12-15,15.0,-0.39
163,911711010005,anika,2017-12-30,2017-12-16,2017-12-16,12.0,-0.46
183,912031010017,july,2017-12-18,2017-11-17,2017-12-17,11.0,-0.03
192,587480010002,Asma,2017-12-18,2017-10-14,2017-12-17,12.0,-0.03


In [18]:
before_second_age_clean = len(df)

df = df[
    (df["Age_Months"] >= 0) &
    (df["Age_Months"] <= 120)
].copy()

after_second_age_clean = len(df)

print("Records before second age validation:", before_second_age_clean)
print("Records after second age validation:", after_second_age_clean)
print("Removed records:", before_second_age_clean - after_second_age_clean)


Records before second age validation: 7889
Records after second age validation: 7824
Removed records: 65


## 16. Duplicate Visit Handling

If the same child has multiple records on the same GMP date, those records are merged.

For MUAC and age values, median is used.


In [19]:
before_duplicate_clean = len(df)

agg_dict = {
    "Name": "first",
    "DOB": "first",
    "MUAC": "median",
    "Age": "median",
    "Age_Months": "median",
    "Age_Diff": "median",
    "Name_Original": "first",
    "DOB_Original": "first"
}

df_clean = (
    df.groupby(["ID", "GmpDate"], as_index=False)
    .agg(agg_dict)
)

after_duplicate_clean = len(df_clean)

print("Records before duplicate cleaning:", before_duplicate_clean)
print("Records after duplicate cleaning:", after_duplicate_clean)
print("Removed duplicate visit records:", before_duplicate_clean - after_duplicate_clean)


Records before duplicate cleaning: 7824
Records after duplicate cleaning: 6785
Removed duplicate visit records: 1039


## 17. MUAC-Based Condition Classification

This section creates a nutritional condition label from MUAC.

This is a rule-based classification, not a machine learning model.


In [20]:
def classify_condition(muac):

    if muac < 11.5:
        return "Severe"

    elif muac < 12.5:
        return "Moderate"

    elif muac < 13.5:
        return "At Risk"

    else:
        return "Good"


df_clean["Condition"] = df_clean["MUAC"].apply(classify_condition)

df_clean["Condition"].value_counts()


Condition
Good        3939
At Risk     1651
Moderate     760
Severe       435
Name: count, dtype: int64

## 18. Visit Count Analysis

For condition classification and grouping, all children can be used.

For trend analysis and regression prediction, children need at least two visits.


In [21]:
visit_count = df_clean.groupby("ID").size()

print("Total Children:", len(visit_count))
print("Children with only 1 visit:", (visit_count == 1).sum())
print("Children with 2+ visits:", (visit_count >= 2).sum())

visit_count.describe()


Total Children: 3041
Children with only 1 visit: 1705
Children with 2+ visits: 1336


count    3041.000000
mean        2.231174
std         2.140500
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max        16.000000
dtype: float64

## 19. Create Trend-Ready Dataset

This dataset contains only children with two or more visits.

It will be used for progress analysis and regression prediction.


In [22]:
trend_ids = visit_count[
    visit_count >= 2
].index

df_trend = df_clean[
    df_clean["ID"].isin(trend_ids)
].copy()

df_trend = df_trend.sort_values(
    ["ID", "GmpDate"]
)

print("Trend Ready Records:", len(df_trend))
print("Trend Ready Children:", df_trend["ID"].nunique())

df_trend.head()


Trend Ready Records: 5080
Trend Ready Children: 1336


,ID,GmpDate,Name,DOB,MUAC,Age,Age_Months,Age_Diff,Name_Original,DOB_Original,Condition
0,581415010001,2017-12-17,ff,2017-12-17,15.0,84.0,0.00,84.00,ff,2017-12-17,Good
1,581415010001,2017-12-28,ff,2017-12-17,15.0,84.0,0.36,83.64,hdkdj,2017-12-28,Good
2,581415010002,2017-12-17,gg,2017-12-17,13.0,84.0,0.00,84.00,rahim,2017-12-17,At Risk
3,581415010002,2017-12-28,gg,2017-12-17,14.0,84.0,0.36,83.64,ldjjd,2017-12-28,Good
5,581415010004,2017-12-18,hskkd,2017-12-18,13.0,84.0,0.00,84.00,mina,2017-12-18,At Risk


## 20. Final Data Quality Audit

This section checks whether major data quality issues remain after cleaning.


In [23]:
remaining_name_conflict = (
    df_clean.groupby("ID")["Name"]
    .nunique(dropna=True)
    .gt(1)
    .sum()
)

remaining_dob_conflict = (
    df_clean.groupby("ID")["DOB"]
    .nunique(dropna=True)
    .gt(1)
    .sum()
)

remaining_duplicate_visits = (
    df_clean.duplicated(
        subset=["ID", "GmpDate"]
    ).sum()
)

print("Remaining Name Conflicts:", remaining_name_conflict)
print("Remaining DOB Conflicts:", remaining_dob_conflict)
print("Remaining Duplicate Visits:", remaining_duplicate_visits)


Remaining Name Conflicts: 0
Remaining DOB Conflicts: 0
Remaining Duplicate Visits: 0


## 21. Child-Level Audit Example

This section shows before-cleaning and after-cleaning values for a selected child.

It helps verify whether name and DOB standardization worked correctly.


In [24]:
child_id = "581415010001"

available_columns = [
    "ID",
    "GmpDate",
    "Name_Original",
    "Name",
    "DOB_Original",
    "DOB",
    "MUAC",
    "Age",
    "Age_Months",
    "Age_Diff",
    "Condition"
]

available_columns = [
    col for col in available_columns
    if col in df_clean.columns
]

child_info = (
    df_clean[df_clean["ID"] == child_id]
    .sort_values("GmpDate")
)

if len(child_info) == 0:
    print("No records found for this Child ID.")
    print("Sample IDs:")
    print(df_clean["ID"].drop_duplicates().head(10).tolist())
else:
    display(child_info[available_columns])


,ID,GmpDate,Name_Original,Name,DOB_Original,DOB,MUAC,Age,Age_Months,Age_Diff,Condition
0,581415010001,2017-12-17,ff,ff,2017-12-17,2017-12-17,15.0,84.0,0.00,84.00,Good
1,581415010001,2017-12-28,hdkdj,ff,2017-12-28,2017-12-17,15.0,84.0,0.36,83.64,Good


## 22. Select Final Columns and Save Datasets

Two datasets are saved:

1. `cleaned_data.csv` - all clean records with audit columns
2. `trend_ready_data.csv` - children with at least two visits for trend and regression analysis


In [25]:
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = OUTPUT_DIR / "cleaned_data.csv"
trend_path = OUTPUT_DIR / "trend_ready_data.csv"

final_columns = [
    "ID",
    "GmpDate",
    "Name_Original",
    "Name",
    "DOB_Original",
    "DOB",
    "MUAC",
    "Age",
    "Age_Months",
    "Age_Diff",
    "Condition"
]

final_columns = [
    col for col in final_columns
    if col in df_clean.columns
]

clean_output = df_clean[final_columns].copy()
trend_output = df_trend[final_columns].copy()

clean_output.to_csv(cleaned_path, index=False)
trend_output.to_csv(trend_path, index=False)

print("Clean dataset saved:", cleaned_path)
print("Trend ready dataset saved:", trend_path)


Clean dataset saved: ..\data\cleaned_data.csv
Trend ready dataset saved: ..\data\trend_ready_data.csv


## 23. Final Cleaning Summary

This summary shows the final dataset size after cleaning.


In [26]:
summary = {
    "Raw Records": raw_records,
    "Raw Unique Children": raw_children,
    "Final Clean Records": len(clean_output),
    "Final Clean Children": clean_output["ID"].nunique(),
    "Trend Ready Records": len(trend_output),
    "Trend Ready Children": trend_output["ID"].nunique(),
    "Remaining Name Conflicts": remaining_name_conflict,
    "Remaining DOB Conflicts": remaining_dob_conflict,
    "Remaining Duplicate Visits": remaining_duplicate_visits
}

summary


{'Raw Records': 10011,
 'Raw Unique Children': 3550,
 'Final Clean Records': 6785,
 'Final Clean Children': 3041,
 'Trend Ready Records': 5080,
 'Trend Ready Children': 1336,
 'Remaining Name Conflicts': np.int64(0),
 'Remaining DOB Conflicts': np.int64(0),
 'Remaining Duplicate Visits': np.int64(0)}